# When does a quantum computer break your keys?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/benchmarks/cryptography-resource-estimate.ipynb)

Reproduces the estimates published at [zksf.org/applications/cryptography](https://zksf.org/applications/cryptography/).

No head-to-head here: nothing classical competes with Shor's algorithm, and that gap is the point. The question is how large a quantum computer would have to be. Change the key sizes to your own and re-run.


## Surface-code overhead

Logical error per round falls as roughly `0.1 (p/p_th)^(d/2)` for physical error rate `p`, threshold `p_th` near 1 percent, and code distance `d`. Each logical qubit costs about `2d^2` physical ones.

At or above threshold, error correction does not work at all and no number of qubits fixes it. That is the single most important row in the table below.

In [ ]:
P_TH, TARGET, D_MAX = 0.01, 1e-15, 201

def distance(p):
    if p >= P_TH: return None          # error correction cannot help at/above threshold
    d = 3
    while d <= D_MAX and 0.1 * (p / P_TH) ** (d / 2) > TARGET: d += 2
    return d if d <= D_MAX else None

print(f"{'physical error':>16}{'distance':>10}{'physical per logical':>22}")
for p in (1e-2, 5e-3, 1e-3, 1e-4):
    d = distance(p)
    print(f"{p:>16.0e}{(str(d) if d else 'n/a'):>10}"
          f"{(f'{2*d*d:,}' if d else 'never'):>22}")

## What each key costs to break

Shor against an n-bit RSA modulus needs about `2n+3` logical qubits. Elliptic curve discrete logs over an n-bit prime field need roughly `9n`, which is why a 256-bit curve is a **smaller** target than RSA-2048 despite comparable classical security.

Edit `KEYS` for your own inventory.

In [ ]:
P_PHYS = 1e-3          # roughly today's best devices
TODAY   = 108          # largest QPU on the ZKSF platform
d = distance(P_PHYS); ov = 2 * d * d

KEYS = [("RSA-2048", 2*2048+3), ("RSA-4096", 2*4096+3),
        ("ECC P-256 / secp256k1", 9*256), ("ECC P-384", 9*384)]

print(f"at p={P_PHYS}, d={d}, {ov:,} physical per logical\n")
print(f"{'target':>24}{'logical':>10}{'physical':>16}{'vs today':>14}")
for name, logical in KEYS:
    phys = logical * ov
    print(f"{name:>24}{logical:>10,}{phys:>16,}{phys/TODAY:>13,.0f}x")

## Reading this honestly

**Treat these as lower bounds, not forecasts.** The estimate counts data qubits and surface-code overhead. It omits the magic-state distillation factories a real implementation needs for non-Clifford gates, which published full analyses find dominate the footprint and push the figure into the tens of millions. Every omission points the same way: the true requirement is higher.

**Elliptic curve keys fall first.** Any migration plan treating RSA as urgent and elliptic curve as comfortable has the ordering backwards.

**Track fidelity, not qubit count.** Change `P_PHYS` above from 1e-3 to 1e-4 and watch the physical requirement drop by nearly a factor of four. That sensitivity is why credible timelines disagree by decades, and why a migration plan should be indexed to fidelity milestones rather than to a date.

## Next

- [The full benchmark page](https://zksf.org/applications/cryptography/), with the analysis and the caveats
- [How we benchmark](https://zksf.org/applications/methodology/): the rules every one of these follows
- [All applications](https://zksf.org/applications/) across six sectors
- [Certification](https://zksf.org/quantum-computing-certification/): what the accuracy statements assert
